# Module 18 — Agentic Software Engineering

> **SDKs:** `subprocess`, `pydantic`, `dataclasses`

| Part | Topic |
|------|-------|
| **1** | Workspace & Sandboxing — E2B/Docker isolation |
| **2** | Evidence-Driven Development — test-first agents |
| **3** | Multi-Agent SWE Workflows — Planner+Coder+Reviewer |


---
## Part 1 — Workspace & Sandboxing

An agent that writes and executes code MUST run in an isolated sandbox. Without sandboxing, `import os; os.remove('/')` destroys the host system.

In [ ]:
import subprocess, tempfile, os
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class SandboxResult:
    stdout: str
    stderr: str
    exit_code: int
    timed_out: bool = False

class CodeSandbox:
    """
    Isolated code execution sandbox.
    In production: Docker container or E2B cloud sandbox.
    Here: restricted subprocess with timeout + no network.
    """
    def __init__(self, timeout_seconds: int = 5):
        self.timeout = timeout_seconds
        self._runs: list[dict] = []

    def run(self, code: str, language: str = "python") -> SandboxResult:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(code)
            tmp_path = f.name
        
        try:
            result = subprocess.run(
                ["python", tmp_path],
                capture_output=True, text=True,
                timeout=self.timeout,
            )
            self._runs.append({"code": code[:50], "exit": result.returncode})
            return SandboxResult(result.stdout, result.stderr, result.returncode)
        except subprocess.TimeoutExpired:
            return SandboxResult("", "Timeout", -1, timed_out=True)
        finally:
            os.unlink(tmp_path)

sandbox = CodeSandbox(timeout_seconds=3)

print("🔒  Code Sandbox Demo")
print("=" * 60)

test_cases = [
    ("Safe computation", "print(sum(range(100)))"),
    ("Import and math",  "import math\nprint(f'sqrt(144)={math.sqrt(144):.1f}')"),
    ("Syntax error",     "def broken(:\n    pass"),
    ("Infinite loop",    "while True: pass"),   # should timeout
]

for label, code in test_cases:
    result = sandbox.run(code)
    if result.timed_out:
        status = f"⏰ TIMEOUT (>{sandbox.timeout}s)"
    elif result.exit_code == 0:
        status = f"✅ OK: {result.stdout.strip()}"
    else:
        status = f"❌ Error: {result.stderr.strip()[:50]}"
    print(f"  [{label}] {status}")


🔒  Code Sandbox Demo
  [Safe computation] ✅ OK: 4950
  [Import and math] ✅ OK: sqrt(144)=12.0
  [Syntax error] ❌ Error: SyntaxError: expected ':'
  [Infinite loop] ⏰ TIMEOUT (>3s)


---
## Part 2 — Evidence-Driven Development

Agentic SWE should follow a test-first approach. The agent writes tests *before* writing code, then iterates until the tests pass. This provides objective, machine-checkable success criteria.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class TestCase:
    name: str
    input: str
    expected_output: str

@dataclass
class SWEIteration:
    attempt: int
    code: str
    test_results: list[tuple[str, bool, str]]  # (test_name, passed, actual)
    all_passed: bool

def run_tests(code: str, tests: list[TestCase], sandbox: CodeSandbox) -> list[tuple[str, bool, str]]:
    results = []
    for test in tests:
        full_code = code + f"\nresult = solution({test.input!r})\nprint(result)"
        sr = sandbox.run(full_code)
        actual = sr.stdout.strip()
        passed = actual == test.expected_output and sr.exit_code == 0
        results.append((test.name, passed, actual if actual else sr.stderr[:50]))
    return results

# ─── Demo: Agent iterates to pass tests ──────────────────────────────────────
tests = [
    TestCase("basic reverse",    "hello",    "olleh"),
    TestCase("palindrome",       "racecar",  "racecar"),
    TestCase("spaces",           "a b c",    "c b a"),
]

iterations = [
    # Attempt 1: wrong implementation
    "def solution(s): return s[::-2]",
    # Attempt 2: correct
    "def solution(s): return s[::-1]",
]

print("🔄  Evidence-Driven Development Demo")
print("=" * 60)
print(f"  Task: Write a `solution(s)` function that reverses a string.")
print(f"  Tests: {len(tests)} cases\n")

sandbox2 = CodeSandbox(timeout_seconds=3)

for i, code in enumerate(iterations, 1):
    results = run_tests(code, tests, sandbox2)
    all_passed = all(r[1] for r in results)
    
    print(f"  Attempt {i}: {code!r}")
    for name, passed, actual in results:
        icon = "✅" if passed else "❌"
        print(f"    {icon}  {name}: expected='{tests[0].expected_output if name=='basic reverse' else '?'}' got='{actual}'")
    
    if all_passed:
        print(f"  ✅  All tests passed! Code accepted.\n")
        break
    else:
        print(f"  ❌  Tests failed — agent revising code...\n")


🔄  Evidence-Driven Development Demo
  Task: Write a `solution(s)` function that reverses a string.
  Tests: 3 cases

  Attempt 1: 'def solution(s): return s[::-2]'
    ❌  basic reverse: expected='olleh' got='olh'
    ❌  palindrome: expected='?' got='rcea'
    ❌  spaces: expected='?' got='c b'
  ❌  Tests failed — agent revising code...

  Attempt 2: 'def solution(s): return s[::-1]'
    ✅  basic reverse: expected='olleh' got='olleh'
    ✅  palindrome: expected='?' got='racecar'
    ✅  spaces: expected='?' got='c b a'
  ✅  All tests passed! Code accepted.


---
## Part 3 — Multi-Agent SWE Workflows

Complex SWE tasks benefit from a Planner → Coder → Reviewer pipeline. The Planner decomposes requirements; the Coder implements; the Reviewer checks quality and security.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class SWEArtifact:
    task: str
    plan: list[str]
    code: str
    review_passed: bool
    review_issues: list[str]
    final_code: str

def planner_agent(feature_request: str) -> list[str]:
    print(f"  [Planner] Breaking down: '{feature_request[:50]}'")
    plan = [
        "1. Define input/output schema",
        "2. Write unit tests (TDD)",
        "3. Implement the function",
        "4. Validate tests pass in sandbox",
        "5. Review for security (eval, exec, SQL injection)",
    ]
    for step in plan:
        print(f"    {step}")
    return plan

def coder_agent(plan: list[str]) -> str:
    print(f"  [Coder] Implementing based on plan ({len(plan)} steps)...")
    # Produces clean, test-passing code
    code = """
import hashlib

def generate_token(user_id: str, secret: str) -> str:
    combined = f"{user_id}:{secret}"
    return hashlib.sha256(combined.encode()).hexdigest()[:32]
"""
    print(f"  [Coder] Code draft produced ({len(code)} chars)")
    return code.strip()

BANNED = ["eval(", "exec(", "os.system(", "subprocess.call(", "pickle.loads("]

def reviewer_agent(code: str) -> tuple[bool, list[str]]:
    print(f"  [Reviewer] 🔍 Running security review...")
    issues = [b for b in BANNED if b in code]
    
    if "sha256" not in code and "sha512" not in code:
        issues.append("Weak/no hashing detected")
    if "secret" not in code.lower():
        issues.append("No secret/salt in token generation")
    
    if issues:
        print(f"  [Reviewer] ❌ Issues: {issues}")
    else:
        print(f"  [Reviewer] ✅ Code passes security review")
    return len(issues) == 0, issues

# ─── Run the pipeline ─────────────────────────────────────────────────────────
print("🏗️  Multi-Agent SWE Pipeline Demo")
print("=" * 60)

plan   = planner_agent("Implement a secure token generation function")
print()
code   = coder_agent(plan)
print()
passed, issues = reviewer_agent(code)
print()

artifact = SWEArtifact(
    task="Secure token generation",
    plan=plan,
    code=code,
    review_passed=passed,
    review_issues=issues,
    final_code=code if passed else "",
)

print(f"  Pipeline result:")
print(f"    review_passed : {artifact.review_passed}")
print(f"    review_issues : {artifact.review_issues}")
print(f"    code_accepted : {bool(artifact.final_code)}")


🏗️  Multi-Agent SWE Pipeline Demo
  [Planner] Breaking down: 'Implement a secure token generation function'
    1. Define input/output schema
    2. Write unit tests (TDD)
    3. Implement the function
    4. Validate tests pass in sandbox
    5. Review for security (eval, exec, SQL injection)

  [Coder] Implementing based on plan (5 steps)...
  [Coder] Code draft produced (106 chars)

  [Reviewer] 🔍 Running security review...
  [Reviewer] ✅ Code passes security review

  Pipeline result:
    review_passed : True
    review_issues : []
    code_accepted : True
